# Rough-Heston calibration study — piecewise-constant chaos

Run 100 independent fits using the common settings from rough-Heston Legendre, with the piecewise time grid and quadrature rules retained. Each run fits
Heston to calibration quotes, projects fresh starting coefficients, and calibrates
the chaos model. Initialization/training seeds vary; comparison/test seeds match
Legendre and remain fixed. Quiet reporting and the study run count are retained.

Use **Restart Kernel → Run All** to run or resume the aligned configuration.


## 1. Imports and data

Load project libraries and the calibration/test surfaces. Paths work from the repository or the notebook directory.

In [ ]:
from pathlib import Path
import importlib
import json
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm import tqdm

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "wiener_chaos/core/model.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")
STUDY_DIR = ROOT / "Rough Heston/Piecewise constant"
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

from wiener_chaos.core import model as pricing_module
pricing_module = importlib.reload(pricing_module)
from wiener_chaos.experiments import calibration as study
study = importlib.reload(study)
print(f"Study workflow loaded: {study.__file__}")

DATA_PATH = ROOT / "Rough Heston/rough_heston_data.pkl"
with DATA_PATH.open("rb") as handle:
    data = pickle.load(handle)

pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})


## 2. Settings aligned with rough-Heston Legendre

The shared configuration matches `../Legendre/Calibration.ipynb`:
`P=3`, `d=2`, tensor backend, 32,768-path chunks, `lr=1e-4`, at most 600 steps,
patience 60, `tol_error=2e-8`, `limit_error_above=2e-6`, and the same scheduler.
Training draws refresh every 50 steps. Degree-2 moment control variates are active
immediately, all samples estimate their weights, and both regularization weights
are zero. Stopping monitors training loss; validation monitoring is disabled.

Heston uses the same surface-derived starting guess, optimizer options, plain
projection without antithetic paths, 5,000 × 128 projection paths, 201 time points,
and one projection repetition. Calibration quotes alone determine initialization.

The ten-interval piecewise grid is retained. Calibration pricing keeps Gauss–Hermite
at its first two maturities with 40 and 25 nodes. Subsequent maturities use
`[200_000] * 2 + [100_000] * 6` MC paths in training and independent evaluation.
Thus there are **1 million MC paths** per calibration-surface pricing pass,
alongside the two quadrature maturities.

Held-out pricing matches the calibration notebook: Gauss–Hermite at its first
two maturities with 50 and 40 nodes. The remaining budgets match Legendre:
`[1_000_000] + [500_000] * 4` MC paths (**3 million** total).
Fixed comparison seeds match Legendre: validation 4,000,042 (inactive),
calibration-surface evaluation 5,000,042, and held-out evaluation 6,000,042.
Printing settings and GPU selection are retained.

### Optional heuristic-only Heston initialization

Leave `CONFIG["initialization"]["fit_heston"] = True` for the current fitted-Heston
warm start. Set it to `False` to use the existing surface-based Heston heuristic
directly, skipping Heston optimization. The chaos projection and subsequent chaos
calibration still run with the same budgets. `heston_fit` is empty when the Heston
optimizer is skipped. Use the same choice in calibration and calibration study.


In [ ]:
CUDA_DEVICE_INDEX = 1  # Index of the GPU used for all model computations.
if torch.cuda.is_available() and CUDA_DEVICE_INDEX >= torch.cuda.device_count():
    raise RuntimeError(f"CUDA device {CUDA_DEVICE_INDEX} is unavailable.")
DEVICE = f"cuda:{CUDA_DEVICE_INDEX}" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    # Small tensor operations are faster without a large CPU thread pool.
    torch.set_num_threads(1)


CONFIG = {
    # Master seed for reproducible initialization and sampling.
    "base_seed": 42,
    # Device used for model tensors and pricing.
    "device": DEVICE,
    # Algorithm for evaluating conditional chaos moments.
    "backend": "tensor",
    # Number of Brownian drivers.
    "d": 2,
    # Maximum total chaos order.
    "P": 3,
    # Time-grid endpoints for the piecewise Brownian basis.
    "pi_M": [0, 0.019178, 0.038356, 0.082192, 0.164274, 0.246411, 0.328548, 0.492823, 0.684476, 1, 1.5],
    # Paths evaluated per chunk by the general-basis pricing kernel.
    "pricing_chunk_size": 32_768,
    # How the starting chaos coefficients are obtained.
    "initialization": {
        # Choose a fitted-Heston projection or a random chaos start.
        "type": "calibrate_heston",
        # Starting point for the Heston parameter optimizer.
        "initial_guess": "surface",
        # False uses the surface heuristic directly, without Heston optimization.
        "fit_heston": True,
        # Same plain projection estimator as Legendre.
        "projection_control": "none",
        # Match Legendre by leaving antithetic projection disabled.
        "projection_antithetic": False,
        # Independent projection estimates averaged before normalization.
        "projection_repetitions": 1,
        # Brownian paths in each terminal-factor projection batch.
        "N": 5_000,
        # Number of projection batches; total paths are N times Ns.
        "Ns": 128,
        # Number of time-grid points in each Heston simulation.
        "n_steps_heston": 201,
        # Settings passed to the Heston optimizer.
        "options": {
            # Maximum Heston optimizer iterations.
            "maxiter": 500,
            # Heston optimizer tolerance for objective improvement.
            "ftol": 1e-12,
            # Heston optimizer tolerance for the gradient.
            "gtol": 1e-9,
        },
    },
    # Pricing allocations at calibration maturities.
    "pricing_calibr": {
        # Integration method for each calibration maturity (MC or Gauss-Hermite).
        "method": ['Gauss_Hermite', 'Gauss_Hermite', 'MC', 'MC', 'MC', 'MC', 'MC', 'MC', 'MC', 'MC'],
        # MC paths per calibration maturity; None for quadrature.
        # More paths at the shortest maturity to resolve deep OTM payoffs.
        "N": [None, None, 200000, 200000, 100000, 100000, 100000, 100000, 100000, 100000],
        # Independent MC batches per calibration maturity.
        "Ns": [1] * 10,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [40, 25, None, None, None, None, None, None, None, None],
    },
    # MC paths per maturity for fixed validation and evaluation samples.
    "comparison_N_per_mat": [None, None, 200000, 200000, 100000, 100000, 100000, 100000, 100000, 100000],
    # Optimizer, control-variate, and stopping settings.
    "calibration": {
        # Maximum number of optimizer steps.
        "n_iter": 600,
        # Initial AdamW learning rate.
        "lr": 1e-4,
        # Highest stock-price power used as an MC control variate.
        "poly_cv_degree": 2,
        # Non-improving monitoring checks allowed before early stopping.
        "patience": 60,
        # Minimum monitored improvement needed to reset patience.
        "tol_error": 2e-8,
        # Optimizer steps before activating MC control variates.
        "warmup_without_cv": 0,
        # Best monitored loss must be below this to stop early.
        "limit_error_above": 2e-6,
        # Steps between fresh training MC draws; None keeps draws fixed.
        "simulate_every": 50,
        # AdamW decoupled weight-decay coefficient.
        "reg_lambda": 0,
        # Weight of the penalty toward the starting chaos coefficients.
        "anchor_lambda": 0,
        # MC paths reserved to estimate control-variate weights.
        #"n_samples_cv": 10_000,
        # Steps between fixed validation checks; None monitors training loss.
        "check_validation_set_MC": None,
        # Steps between price-loss and speed reports; None suppresses reports.
        "print_loss_every_it": None,
        # Whether model.calibrate displays fit plots.
        "print_plots": False,
        # Learning-rate scheduler settings for stalled monitoring loss.
        "plateau_kwargs": {
            # Whether lower or higher monitored values are better.
            "mode": "min",
            # Multiplier applied to the learning rate on a plateau.
            "factor": 0.95,
            # Non-improving checks before reducing the learning rate.
            "patience": 15,
            # Minimum change counted as scheduler improvement.
            "threshold": 1e-7,
            # Interpret threshold as absolute or relative.
            "threshold_mode": "abs",
            # Checks to wait after a learning-rate reduction.
            "cooldown": 0,
            # Lowest learning rate allowed by the scheduler.
            "min_lr": 1e-6,
            # Smallest learning-rate change worth applying.
            "eps": 1e-8,
        },
    },
    # Held-out pricing retains the piecewise quadrature rules.
    "pricing_test": {'method': ['Gauss_Hermite', 'Gauss_Hermite', 'MC', 'MC', 'MC', 'MC', 'MC'], 'N': [None, None, 1000000, 500000, 500000, 500000, 500000], 'Ns': [1, 1, 1, 1, 1, 1, 1], 'nodes': [30, 15, None, None, None, None, None]},
}

FIXED_VALIDATION_SEED = 4_000_042
FIXED_EVALUATION_SEED = 5_000_042
FIXED_TEST_SEED = 6_000_042

CONFIG["n_calibrations"] = 100

OUTPUT_BASE_DIR = STUDY_DIR / "Statistics calibration" / "study"
STUDY_OPTIONS = {
    "fixed_validation_seed": FIXED_VALIDATION_SEED,
    "fixed_evaluation_seed": FIXED_EVALUATION_SEED,
    "fixed_test_seed": FIXED_TEST_SEED,
}
print(f"Device: {DEVICE} | Calibrations: {CONFIG['n_calibrations']}")
print(f"Results base directory: {OUTPUT_BASE_DIR}")

print(f"Training MC paths: {sum(n or 0 for n in CONFIG['pricing_calibr']['N']):,} | "
      f"Comparison MC paths: {sum(n or 0 for n in CONFIG['comparison_N_per_mat']):,}")
maturity_table = pd.DataFrame([
    {"set": label, "maturity": t, "pricing": method, "N": count, "nodes": nodes}
    for label, key in (("Calibration", "calibr"), ("Test", "test"))
    for t, method, count, nodes in zip(
        data[f"maturities_{key}"], CONFIG[f"pricing_{key}"]["method"],
        CONFIG[f"pricing_{key}"]["N"], CONFIG[f"pricing_{key}"]["nodes"],
    )
]).sort_values("maturity", ignore_index=True)
display(maturity_table)

# Reuse completed studies automatically; set False to run or resume calibrations.
SAVED_RESULTS_DIR = OUTPUT_BASE_DIR
LOAD_SAVED_RESULTS = (study.resolve_study_directory(SAVED_RESULTS_DIR) / "results.json").is_file()


## 3. Study workflow

Run, checkpoint, and loading functions are in [calibration.py](../../wiener_chaos/experiments/calibration.py). Data, hyperparameters, and CUDA device are set here independently.


## 4. Run or load the study

Load the latest completed study automatically when available. Otherwise, run
or resume the configured calibrations. Set `LOAD_SAVED_RESULTS=False` explicitly
to run or resume calibrations. Completed runs are checkpointed, and a completed
study becomes the default for analysis and plotting automatically.


In [ ]:
if LOAD_SAVED_RESULTS:
    if SAVED_RESULTS_DIR is None:
        raise ValueError("Set SAVED_RESULTS_DIR to a completed study folder before loading.")
    records, OUTPUT_DIR, ANALYSIS_CONFIG = study.load_synthetic_results(SAVED_RESULTS_DIR)
    print(f"Loaded {len(records)} saved calibrations from {OUTPUT_DIR}")
else:
    with tqdm(total=CONFIG["n_calibrations"], desc="Rough Heston calibrations", unit="calibration") as progress_bar:
        records, OUTPUT_DIR = study.run_study(
            data, CONFIG, OUTPUT_BASE_DIR, DATA_PATH,
            progress=study.tqdm_progress_callback(progress_bar), **STUDY_OPTIONS,
        )
    ANALYSIS_CONFIG = CONFIG
    print(f"Completed {len(records)} calibrations. Results saved in: {OUTPUT_DIR}")
runs = study.results_tables(records)

runs["reached_iteration_cap"] = [r.get("reached_iteration_cap") for r in records]
runs["test_evaluation_s"] = [r.get("elapsed_seconds_test_evaluation") for r in records]


## 5. Accuracy and runtime

Compare IV MAE at calibration and held-out maturities. Initialization includes
Heston fitting and projection; calibration includes optimization and independent
final calibration-surface pricing. Their sum excludes held-out evaluation and
reporting. Comparison seeds are fixed across runs as in Legendre, while
projection and training draws vary. Invalid IVs invalidate the corresponding MAE.


In [ ]:
run_table = runs.set_index("run").copy()
run_table["initialization_s"] = run_table["elapsed_seconds_init"]
run_table["calibration_s"] = run_table["elapsed_seconds_chaos_calibration"]
run_table["total_s"] = run_table["initialization_s"] + run_table["calibration_s"]
quality_columns = ["iv_mae_calibration_bp", "iv_mae_test_bp", "loss_ins", "loss_oos",
                   "invalid_iv_calibration", "invalid_iv_test",
]
fit_summary = run_table[quality_columns].agg(["count", "mean", "std", "min", "max"]).T
timing_summary = run_table[["initialization_s", "calibration_s", "total_s"]].agg(
    ["count", "mean", "std", "min", "max"]
).T
display(fit_summary)
display(timing_summary)
print(f"Sum of initialization times: {run_table['initialization_s'].sum():.2f}s | "
      f"Sum of calibration times: {run_table['calibration_s'].sum():.2f}s | "
      f"Combined: {run_table['total_s'].sum():.2f}s")
print(f"Runs with invalid calibration IVs: {int((run_table['invalid_iv_calibration'] > 0).sum())}")
display(run_table[["calibration_iterations", "reached_iteration_cap", "iv_mae_calibration_bp", "iv_mae_test_bp"]])
print(f"Runs reaching the iteration cap: {int(run_table['reached_iteration_cap'].eq(True).sum())}")


In [ ]:
figures = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, column in (("Calibration", "iv_mae_calibration_bp"), ("Test", "iv_mae_test_bp")):
    axes[0].plot(run_table.index, run_table[column], ".", label=label)
axes[0].set(xlabel="Calibration run", ylabel="IV MAE (bp)", title="Fit quality across runs")
axes[0].legend()
run_table[["initialization_s", "calibration_s", "total_s"]].plot(ax=axes[1])
axes[1].set(xlabel="Calibration run", ylabel="Seconds", title="Initialization and calibration time")
fig.tight_layout()
plt.show()
figures["study_summary"] = fig

## 6. Saved results

The study folder contains `manifest.json` and aggregate `results.json`, with
one checkpoint per calibration in `runs/run_XXX/result.json`. Coefficient tensors
are saved in `coefficients/`; fit and timing tables in `tables/`; summary plots
in `figures/`. Additional coefficient diagnostics are grouped in `diagnostics/`.
The relative `study_latest.json` pointer selects the completed study for loading
and plotting automatically.


In [ ]:
study.save_artifacts(
    OUTPUT_DIR,
    coefficients={"initial_coefficients": torch.tensor([r["initial_coefficients"] for r in records], dtype=torch.float32),
                  "calibrated_coefficients": torch.tensor([r["d_a"] for r in records], dtype=torch.float32)},
    tables={"calibrations": run_table, "fit_summary": fit_summary, "timing_summary": timing_summary},
    figures=figures,
)
print(f"Study results saved in: {OUTPUT_DIR}")
